In [3]:
import feedparser
import pandas as pd
from datetime import datetime, timezone
import urllib.parse
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

# =============================================================
# STEP 1: DEFINE AUSTRALIAN RETAIL BRANDS & KEYWORDS
# =============================================================
retail_brands = [
    "Woolworths", "Coles", "ALDI", "Kmart", 
    "Bunnings", "JB Hi-Fi", "Big W", "IGA", "Amazon Australia"
]

# Industry-specific keywords to ensure headline relevance
retail_keywords = [
    "supermarket", "grocery", "groceries", "retail", "price", 
    "store", "inflation", "sales", "discount", "shopping", 
    "supply chain", "deals", "profit", "checkout", "liquor"
]

# Pure retail/supermarket players with higher keyword tolerance
pure_retail_brands = ["Woolworths", "Coles", "ALDI", "Kmart", "Bunnings", "Big W", "IGA"]

all_articles = []

print("Fetching Australian Retail RSS feeds...")
for brand in retail_brands:
    # Google News search query anchored in Australia
    search_term = f'"{brand}" (supermarket OR grocery OR retail OR store OR price OR sales)'
    encoded_query = urllib.parse.quote(search_term)
    
    rss_url = f"https://news.google.com/rss/search?q={encoded_query}&hl=en-AU&gl=AU&ceid=AU:en"
    feed = feedparser.parse(rss_url)
    
    matched_count = 0
    for entry in feed.entries:
        title_lower = entry.title.lower()
        
        # Include if pure retail brand OR headline matches retail keywords
        is_relevant = (brand in pure_retail_brands) or any(keyword in title_lower for keyword in retail_keywords)
        
        if is_relevant:
            all_articles.append({
                "Brand": brand,
                "Title": entry.title,
                "Published": entry.published,
                "Link": entry.link
            })
            matched_count += 1
            
    print(f"  -> {brand}: Found {matched_count} articles (out of {len(feed.entries)} returned)")

# =============================================================
# STEP 2: CREATE DATAFRAME & CALCULATE RECENCY
# =============================================================
master_df = pd.DataFrame(all_articles)

if not master_df.empty:
    master_df.to_csv("master_retail_news.csv", index=False)
    print(f"\n✅ Saved {len(master_df)} total articles to 'master_retail_news.csv'\n")

    # Convert 'Published' string to timezone-aware datetime
    master_df['Published_Date'] = pd.to_datetime(master_df['Published'], format='%a, %d %b %Y %H:%M:%S %Z', errors='coerce')

    current_time = datetime.now(timezone.utc)
    master_df['Days_Ago'] = (current_time - master_df['Published_Date']).dt.days

    # Timeframe boolean flags
    master_df['Last_7_Days'] = master_df['Days_Ago'] <= 7
    master_df['Last_14_Days'] = master_df['Days_Ago'] <= 14
    master_df['Last_30_Days'] = master_df['Days_Ago'] <= 30

    # Group by Brand
    summary_table = master_df.groupby('Brand').agg(
        Total_Articles=('Brand', 'count'),
        Last_7_Days=('Last_7_Days', 'sum'),
        Last_14_Days=('Last_14_Days', 'sum'),
        Last_30_Days=('Last_30_Days', 'sum')
    ).reset_index().sort_values(by='Last_7_Days', ascending=False)

    summary_table.to_csv("retail_recency_summary.csv", index=False)
    print("📊 Retail Article Recency Summary:")
    print(summary_table.to_string(index=False))

    # =============================================================
    # STEP 3: GENERATE CONDITIONAL FORMATTED HEATMAP
    # =============================================================
    heatmap_data = summary_table.set_index('Brand')
    heatmap_data = heatmap_data.drop(columns=['Total_Articles', 'total_articles'], errors='ignore')

    # Normalize columns independently for scaling
    normalized_data = heatmap_data.apply(
        lambda col: (col - col.min()) / (col.max() - col.min()) if (col.max() - col.min()) != 0 else col * 0
    )

    color_list = ["#ff4d4d", "#ffdb4d", "#4dff4d"]  
    custom_cmap = mcolors.LinearSegmentedColormap.from_list("RedYellowGreen", color_list)

    plt.figure(figsize=(10, 8))
    sns.heatmap(
        normalized_data, 
        annot=heatmap_data,      
        fmt="d",                 
        cmap=custom_cmap, 
        linewidths=1.0, 
        linecolor="#ffffff",
        cbar=False               
    )

    plt.title('Australian Retail Brand Article Recency Heatmap', fontsize=14, pad=20, weight='bold')
    plt.ylabel('Retail Brand', fontsize=12, labelpad=10)
    plt.xlabel('Recency Windows', fontsize=12, labelpad=10)

    plt.xticks(rotation=0)
    plt.yticks(rotation=0)

    plt.tight_layout()
    output_image_path = "retail_recency_heatmap.png"
    plt.savefig(output_image_path, dpi=300)
    plt.close()

    print(f"\n📊 Process complete. Heatmap saved to '{output_image_path}'!")
else:
    print("❌ No matching articles were found.")

Fetching Australian Retail RSS feeds...
  -> Woolworths: Found 100 articles (out of 100 returned)
  -> Coles: Found 100 articles (out of 100 returned)
  -> ALDI: Found 100 articles (out of 100 returned)
  -> Kmart: Found 100 articles (out of 100 returned)
  -> Bunnings: Found 100 articles (out of 100 returned)
  -> JB Hi-Fi: Found 69 articles (out of 100 returned)
  -> Big W: Found 100 articles (out of 100 returned)
  -> IGA: Found 100 articles (out of 100 returned)
  -> Amazon Australia: Found 61 articles (out of 100 returned)

✅ Saved 830 total articles to 'master_retail_news.csv'

📊 Retail Article Recency Summary:
           Brand  Total_Articles  Last_7_Days  Last_14_Days  Last_30_Days
            ALDI             100           36            39            47
           Coles             100           30            32            45
      Woolworths             100           19            22            29
           Big W             100            7             8            11
     